In [2]:
"""
Corporate Actions Module
─────────────────────────
Automated discovery of NSE corporate actions:
  • Fetches NSE corporate actions API (last 30 + next 30 days)
  • Classifies each action via keyword parsing
  • Persists to corporate_actions.json
  • Flags risky events (DEMERGER, MERGER, RENAME) for user review
  • Does NOT auto-quarantine — flags only, user decides

User decision (logged in README): Flag, don't quarantine.
"""

import json
import os
import re
import requests
from datetime import datetime, timedelta
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CORP_ACTIONS_FILE = "corporate_actions.json"
QUARANTINE_DAYS   = 60   # Used only if user manually decides to quarantine
LOOKBACK_DAYS     = 30   # How far back to fetch corp actions
LOOKAHEAD_DAYS    = 30   # How far forward to fetch

NSE_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Accept":     "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":    "https://www.nseindia.com/companies-listing/corporate-filings-actions",
    "Connection": "keep-alive",
}


# ─────────────────────────────────────────────
# CLASSIFICATION (keyword-based)
# ─────────────────────────────────────────────
# Ordered: more specific patterns checked before general ones
CLASSIFICATION_RULES = [
    # (regex pattern, action_type, is_risky)
    (r"\bdemerg(er|ed|ing)\b",                                 "DEMERGER", True),
    (r"\bscheme of arrangement\b",                             "DEMERGER", True),
    (r"\b(merger|amalgamation|acquisition)\b",                 "MERGER",   True),
    (r"\b(name change|change of name|renamed|new name)\b",     "RENAME",   True),
    (r"\b(spin[- ]?off|hive[- ]?off)\b",                       "DEMERGER", True),
    (r"\bstock split\b",                                       "SPLIT",    False),
    (r"\bsub[- ]?division\b",                                  "SPLIT",    False),
    (r"face value.*(split|sub-?divid|from.*to)",               "SPLIT",    False),
    (r"\bbonus\b",                                             "BONUS",    False),
    (r"\bbuy[- ]?back\b",                                      "BUYBACK",  False),
    (r"\brights\b",                                            "RIGHTS",   False),
    (r"\b(interim|final|special)?\s*dividend\b",               "DIVIDEND", False),
]


def classify_action(text):
    """Classify a corporate action's free-text description."""
    if not text:
        return "UNKNOWN", False
    text_lower = text.lower()
    for pattern, action_type, is_risky in CLASSIFICATION_RULES:
        if re.search(pattern, text_lower):
            return action_type, is_risky
    return "OTHER", False


# ─────────────────────────────────────────────
# FETCH FROM NSE
# ─────────────────────────────────────────────
def fetch_nse_corporate_actions():
    """
    Fetch corporate actions from NSE.
    Returns list of dicts or empty list on failure.
    """
    session = requests.Session()
    session.headers.update(NSE_HEADERS)

    # Prime cookies
    try:
        session.get("https://www.nseindia.com", timeout=10)
        session.get("https://www.nseindia.com/companies-listing/corporate-filings-actions",
                    timeout=10)
    except Exception:
        pass

    today = datetime.now()
    from_date = (today - timedelta(days=LOOKBACK_DAYS)).strftime("%d-%m-%Y")
    to_date   = (today + timedelta(days=LOOKAHEAD_DAYS)).strftime("%d-%m-%Y")

    url = (f"https://www.nseindia.com/api/corporates-corporateActions"
           f"?index=equities&from_date={from_date}&to_date={to_date}")

    try:
        resp = session.get(url, timeout=20)
        if resp.status_code != 200:
            print(f"   ⚠️  NSE API returned status {resp.status_code}")
            return []
        data = resp.json()
        if not isinstance(data, list):
            print(f"   ⚠️  Unexpected NSE API response format")
            return []
        print(f"   ✅ NSE corp actions fetched: {len(data)} entries")
        return data
    except Exception as e:
        print(f"   ⚠️  NSE corp actions fetch failed: {e}")
        return []


# ─────────────────────────────────────────────
# PERSIST TO JSON
# ─────────────────────────────────────────────
def load_existing():
    """Load existing corporate_actions.json."""
    if not Path(CORP_ACTIONS_FILE).exists():
        return {"actions": [], "last_updated": None}
    try:
        with open(CORP_ACTIONS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {"actions": [], "last_updated": None}


def save(data):
    """Save to corporate_actions.json."""
    data["last_updated"] = datetime.now().isoformat()
    with open(CORP_ACTIONS_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# ─────────────────────────────────────────────
# UPDATE WORKFLOW
# ─────────────────────────────────────────────
def update_corporate_actions():
    """
    Main entry point. Fetches, classifies, persists, returns new risky flags.
    Returns: dict with 'new_risky', 'total_actions', 'risky_pending'
    """
    print("\n📋 Updating Corporate Actions...")

    existing = load_existing()
    existing_keys = {
        (a["symbol"], a["ex_date"], a["raw_text"][:50])
        for a in existing.get("actions", [])
    }

    raw_actions = fetch_nse_corporate_actions()
    new_risky_flags = []
    added_count = 0

    for entry in raw_actions:
        symbol  = entry.get("symbol", "").strip()
        company = entry.get("comp", "").strip()
        purpose = entry.get("subject", "").strip() or entry.get("purpose", "").strip()
        ex_date = entry.get("exDate") or entry.get("ex_date", "")
        record  = entry.get("recDate") or entry.get("record_date", "")

        if not symbol or not purpose:
            continue

        # Skip duplicates
        key = (symbol, ex_date, purpose[:50])
        if key in existing_keys:
            continue

        action_type, is_risky = classify_action(purpose)

        action_entry = {
            "symbol":        symbol,
            "company_name":  company,
            "action_type":   action_type,
            "raw_text":      purpose,
            "ex_date":       ex_date,
            "record_date":   record,
            "is_risky":      is_risky,
            "user_decision": "PENDING" if is_risky else "AUTO_OK",
            "quarantine_until": None,
            "discovered_at": datetime.now().isoformat(),
        }
        existing["actions"].append(action_entry)
        added_count += 1

        if is_risky:
            new_risky_flags.append(action_entry)

    save(existing)

    # Summary
    risky_pending = [
        a for a in existing["actions"]
        if a.get("is_risky") and a.get("user_decision") == "PENDING"
    ]

    print(f"   ✅ {added_count} new corp actions added")
    print(f"   ⚠️  {len(new_risky_flags)} new RISKY events (need review)")
    print(f"   📌 {len(risky_pending)} total RISKY events pending decision")

    return {
        "new_risky":     new_risky_flags,
        "total_actions": len(existing["actions"]),
        "risky_pending": risky_pending,
    }


# ─────────────────────────────────────────────
# QUARANTINE HELPERS (used by scanner)
# ─────────────────────────────────────────────
def _parse_ex_date(date_str):
    """
    Parse NSE date strings (commonly '%d-%b-%Y', e.g. '30-Apr-2026').
    Returns datetime.date or None.
    """
    if not date_str:
        return None
    for fmt in ("%d-%b-%Y", "%d-%B-%Y", "%Y-%m-%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(date_str.strip(), fmt).date()
        except Exception:
            continue
    return None


def get_upcoming_actions(days_ahead=7):
    """
    Return all corporate actions with ex_date within the next `days_ahead` days
    (including today). Sorted by ex_date.
    """
    existing = load_existing()
    today = datetime.now().date()
    cutoff = today + timedelta(days=days_ahead)
    upcoming = []

    for action in existing.get("actions", []):
        ex_date = _parse_ex_date(action.get("ex_date"))
        if ex_date and today <= ex_date <= cutoff:
            upcoming.append({
                "symbol":      action["symbol"],
                "company":     action.get("company_name", ""),
                "action_type": action["action_type"],
                "ex_date":     action.get("ex_date", ""),
                "ex_date_obj": ex_date,
                "is_risky":    action.get("is_risky", False),
                "details":     action.get("raw_text", ""),
            })

    upcoming.sort(key=lambda x: x["ex_date_obj"])
    return upcoming


def get_quarantined_symbols():
    """
    Return set of symbols currently quarantined.
    A symbol is quarantined only if user has explicitly set quarantine_until.
    """
    existing = load_existing()
    today = datetime.now().date()
    quarantined = set()

    for action in existing.get("actions", []):
        q_until = action.get("quarantine_until")
        if q_until:
            try:
                q_date = datetime.fromisoformat(q_until).date()
                if q_date >= today:
                    quarantined.add(action["symbol"])
            except Exception:
                pass

    return quarantined


def get_flagged_symbols():
    """
    Return list of symbols with PENDING risky decisions.
    These are FLAGGED but not quarantined per user's preference.
    """
    existing = load_existing()
    flagged = []
    for action in existing.get("actions", []):
        if action.get("is_risky") and action.get("user_decision") == "PENDING":
            flagged.append({
                "symbol":      action["symbol"],
                "action_type": action["action_type"],
                "ex_date":     action.get("ex_date", ""),
                "details":     action.get("raw_text", ""),
            })
    return flagged


def mark_decision(symbol, ex_date, decision, quarantine=False):
    """
    User helper to update a flagged action.
    decision: 'IGNORE', 'QUARANTINE', 'REFRESH'
    """
    existing = load_existing()
    for action in existing["actions"]:
        if action["symbol"] == symbol and action.get("ex_date") == ex_date:
            action["user_decision"] = decision
            if quarantine:
                q_until = datetime.now() + timedelta(days=QUARANTINE_DAYS)
                action["quarantine_until"] = q_until.isoformat()
            break
    save(existing)


# ─────────────────────────────────────────────
# REPORTING
# ─────────────────────────────────────────────
def format_flag_report(flagged):
    """Format flagged actions as a readable report."""
    if not flagged:
        return "\n🟢 No pending corp action flags."

    lines = [f"\n🚩 PENDING CORP ACTION REVIEW ({len(flagged)} items)"]
    lines.append("─" * 60)
    for f in flagged:
        # Use .get() method to safely access 'details' key with a default value
        details = f.get('details', 'No details available')  # Provide default if key missing
        lines.append(
            f"  • {f['symbol']:<15} | {f['action_type']:<10} | "
            f"ex-date: {f['ex_date']:<12} | {details[:60]}"  # Fixed: Use the safely accessed 'details' variable"
        )
    lines.append("─" * 60)
    lines.append("→ Review and update corporate_actions.json if action needed")
    return "\n".join(lines)


# ─────────────────────────────────────────────
# STANDALONE RUN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    result = update_corporate_actions()
    print(format_flag_report(result["risky_pending"]))



📋 Updating Corporate Actions...
   ✅ NSE corp actions fetched: 104 entries
   ✅ 0 new corp actions added
   ⚠️  0 new RISKY events (need review)
   📌 1 total RISKY events pending decision

🚩 PENDING CORP ACTION REVIEW (1 items)
────────────────────────────────────────────────────────────
  • VEDL            | DEMERGER   | ex-date: 30-Apr-2026  | No details available
────────────────────────────────────────────────────────────
→ Review and update corporate_actions.json if action needed
